# Pinecone Embedding Generation for SKU Tagging

This notebook generates embeddings for:
1. **BT-CT Mappings** (`sku-tagging-mappings` index) - Basic Type to Category mappings from Google Sheets
2. **Catalog SKUs** (`sku-tagging-catalog` index) - SKU names with their tags from Catalog_SKUs.csv

Uses **Gemini embedding-2** model for generating embeddings.

In [1]:
# =============================================================================
# CELL 1: Configuration & Setup
# =============================================================================
import os
import json
import pandas as pd
from dotenv import load_dotenv
import google.genai as genai
from pinecone import Pinecone, ServerlessSpec

# Load environment variables
load_dotenv(override=True)

# ---------- API Keys ----------
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')

# ---------- Pinecone Config ----------
MAPPINGS_INDEX = "sku-tagging-mappings"  # BT-CT mappings
CATALOG_INDEX = "sku-tagging-catalog"    # Catalog SKUs

# ---------- Gemini Embedding Model ----------
EMBEDDING_MODEL = "gemini-embedding-2"
EMBEDDING_DIMENSION = 3072  # gemini-embedding-2 outputs 3072 dimensions

# ---------- Google Sheets Config ----------
SPREADSHEET_ID = "1-1DejLMWTf7YbUNKVa84fIiguL1XXb14wKJ-w28yOh4"
SHEET_IDS = {"BT_GK_mappings": "1433148032", "BT_CT_mappings": "1757740042"}

# ---------- Catalog Config ----------
CATALOG_CSV_PATH = os.path.join("Testing Data", "Catalog_SKUs.csv")

# Initialize clients
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)

print("✅ Configuration loaded")
print(f"   Gemini API key: {'set' if GEMINI_API_KEY else 'NOT SET'}")
print(f"   Pinecone API key: {'set' if PINECONE_API_KEY else 'NOT SET'}")
print(f"   Embedding model: {EMBEDDING_MODEL}")
print(f"   Embedding dimension: {EMBEDDING_DIMENSION}")
print(f"   Mappings index: {MAPPINGS_INDEX}")
print(f"   Catalog index: {CATALOG_INDEX}")

✅ Configuration loaded
   Gemini API key: set
   Pinecone API key: set
   Embedding model: gemini-embedding-2
   Embedding dimension: 3072
   Mappings index: sku-tagging-mappings
   Catalog index: sku-tagging-catalog


In [4]:
# =============================================================================
# CELL 2: Helper Functions
# =============================================================================

def get_embedding(text: str) -> list[float]:
    """Generate embedding for a single text using Gemini embedding-2."""
    result = gemini_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[text],
    )
    return result.embeddings[0].values


def get_embeddings_batch(texts: list[str]) -> list[list[float]]:
    """Generate embeddings for multiple texts in one call."""
    result = gemini_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
    )
    return [emb.values for emb in result.embeddings]


def load_google_sheet(spreadsheet_id: str, sheet_gid: str) -> pd.DataFrame:
    """Load a Google Sheet as a pandas DataFrame."""
    url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={sheet_gid}"
    return pd.read_csv(url)


def ensure_index_exists(index_name: str, dimension: int = EMBEDDING_DIMENSION, recreate_if_wrong_dim: bool = True):
    """Create Pinecone index if it doesn't exist. Optionally recreate if dimension mismatch."""
    existing_indexes = {idx.name: idx for idx in pc.list_indexes()}
    
    if index_name in existing_indexes:
        # Check dimension
        existing_dim = existing_indexes[index_name].dimension
        if existing_dim != dimension:
            if recreate_if_wrong_dim:
                print(f"⚠️ Index {index_name} has wrong dimension ({existing_dim} vs {dimension}). Deleting...")
                pc.delete_index(index_name)
                import time
                time.sleep(5)  # Wait for deletion to complete
            else:
                raise ValueError(f"Index {index_name} has dimension {existing_dim}, expected {dimension}")
        else:
            print(f"  ✅ Index already exists: {index_name} (dim={existing_dim})")
            return pc.Index(index_name)
    
    # Create new index
    print(f"Creating index: {index_name} (dim={dimension})...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    
    # Wait for index to be ready
    import time
    print("  Waiting for index to be ready...")
    while True:
        try:
            index = pc.Index(index_name)
            stats = index.describe_index_stats()
            print(f"  ✅ Created index: {index_name}")
            return index
        except:
            time.sleep(2)


print("✅ Helper functions defined")

✅ Helper functions defined


In [5]:
# =============================================================================
# CELL 3: Test Record - BT-CT Mappings Index
# =============================================================================
# Create a single test record in the sku-tagging-mappings index

print("Testing BT-CT Mappings Index...\n")

# Ensure index exists
mappings_index = ensure_index_exists(MAPPINGS_INDEX)

# Test data: one BT-CT mapping
test_bt = "Corn Flakes"
test_category = "Groceries"

# Generate embedding for the Basic Type name
print(f"Generating embedding for: '{test_bt}'...")
embedding = get_embedding(test_bt)
print(f"  ✅ Generated {len(embedding)}-dim embedding")

# Create the vector record
test_record = {
    "id": f"bt_{test_bt.lower().replace(' ', '_')}",
    "values": embedding,
    "metadata": {
        "basic_type": test_bt,
        "category": test_category,
        "type": "bt_ct_mapping"
    }
}

# Upsert to Pinecone
print(f"Upserting to {MAPPINGS_INDEX}...")
mappings_index.upsert(vectors=[test_record])

print(f"\n✅ Test record created in {MAPPINGS_INDEX}!")
print(f"   ID: {test_record['id']}")
print(f"   Basic Type: {test_bt}")
print(f"   Category: {test_category}")
print(f"\n📋 Check Pinecone console to verify the record.")

Testing BT-CT Mappings Index...

⚠️ Index sku-tagging-mappings has wrong dimension (1024 vs 3072). Deleting...
Creating index: sku-tagging-mappings (dim=3072)...
  Waiting for index to be ready...
  ✅ Created index: sku-tagging-mappings
Generating embedding for: 'Corn Flakes'...
  ✅ Generated 3072-dim embedding
Upserting to sku-tagging-mappings...

✅ Test record created in sku-tagging-mappings!
   ID: bt_corn_flakes
   Basic Type: Corn Flakes
   Category: Groceries

📋 Check Pinecone console to verify the record.


In [6]:
# =============================================================================
# CELL 4: Test Record - Catalog SKUs Index
# =============================================================================
# Create a single test record in the sku-tagging-catalog index

print("Testing Catalog SKUs Index...\n")

# Ensure index exists
catalog_index = ensure_index_exists(CATALOG_INDEX)

# Test data: one catalog SKU
test_sku = "Kelloggs Corn Flakes 250g"
test_sku_category = "Groceries"
test_sku_bt = "Corn Flakes"
test_sku_gks = ["Cereal", "Groceries", "Corn Flakes", "Kelloggs Corn Flakes", "Breakfast Cereal"]

# Generate embedding for the SKU name
print(f"Generating embedding for: '{test_sku}'...")
embedding = get_embedding(test_sku)
print(f"  ✅ Generated {len(embedding)}-dim embedding")

# Create the vector record
test_record = {
    "id": f"sku_{hash(test_sku) % 10**8}",  # Simple hash-based ID
    "values": embedding,
    "metadata": {
        "sku_name": test_sku,
        "category": test_sku_category,
        "basic_type": test_sku_bt,
        "generic_keywords": test_sku_gks,
        "type": "catalog_sku"
    }
}

# Upsert to Pinecone
print(f"Upserting to {CATALOG_INDEX}...")
catalog_index.upsert(vectors=[test_record])

print(f"\n✅ Test record created in {CATALOG_INDEX}!")
print(f"   ID: {test_record['id']}")
print(f"   SKU: {test_sku}")
print(f"   Category: {test_sku_category}")
print(f"   Basic Type: {test_sku_bt}")
print(f"   Generic Keywords: {test_sku_gks}")
print(f"\n📋 Check Pinecone console to verify the record.")

Testing Catalog SKUs Index...

⚠️ Index sku-tagging-catalog has wrong dimension (1024 vs 3072). Deleting...
Creating index: sku-tagging-catalog (dim=3072)...
  Waiting for index to be ready...
  ✅ Created index: sku-tagging-catalog
Generating embedding for: 'Kelloggs Corn Flakes 250g'...
  ✅ Generated 3072-dim embedding
Upserting to sku-tagging-catalog...

✅ Test record created in sku-tagging-catalog!
   ID: sku_81362659
   SKU: Kelloggs Corn Flakes 250g
   Category: Groceries
   Basic Type: Corn Flakes
   Generic Keywords: ['Cereal', 'Groceries', 'Corn Flakes', 'Kelloggs Corn Flakes', 'Breakfast Cereal']

📋 Check Pinecone console to verify the record.


---
## Full Data Upload (Run after verifying test records)

After confirming the test records appear in Pinecone, run the cells below to upload all data.

In [9]:
# =============================================================================
# CELL 5: Load BT-CT Mappings from Google Sheets
# =============================================================================

print("Loading BT-CT mappings from Google Sheets...")

bt_ct_df = load_google_sheet(SPREADSHEET_ID, SHEET_IDS['BT_CT_mappings'])
bt_ct_df.columns = [c.strip() for c in bt_ct_df.columns]

# Get column names (first col = Basic Type, second col = Category)
bt_col = bt_ct_df.columns[0]
ct_col = bt_ct_df.columns[1]

print(f"  ✅ Loaded {len(bt_ct_df)} BT-CT mappings")
print(f"  Columns: {bt_col} -> {ct_col}")
display(bt_ct_df.head())

Loading BT-CT mappings from Google Sheets...
  ✅ Loaded 1605 BT-CT mappings
  Columns: Basic Tag Type -> Category Tag Type


,Basic Tag Type,Category Tag Type
0,Staple Pins,Stationery
1,Isotonic Drink,Groceries
2,Led Bulb,Electronics
3,Nail Polish Remover,Cosmetics
4,Skin Care Gel,Personal Care


In [10]:
# =============================================================================
# CELL 6: Upload All BT-CT Mappings to Pinecone
# =============================================================================
import time

print(f"Uploading {len(bt_ct_df)} BT-CT mappings to {MAPPINGS_INDEX}...\n")

mappings_index = ensure_index_exists(MAPPINGS_INDEX)

BATCH_SIZE = 50  # Process in batches for efficiency
total_uploaded = 0
errors = []

for batch_start in range(0, len(bt_ct_df), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(bt_ct_df))
    batch_df = bt_ct_df.iloc[batch_start:batch_end]
    
    # Prepare texts for embedding
    basic_types = batch_df[bt_col].tolist()
    categories = batch_df[ct_col].tolist()
    
    try:
        # Generate embeddings for all Basic Types in batch
        embeddings = get_embeddings_batch(basic_types)
        
        # Create vector records
        vectors = []
        for i, (bt, cat, emb) in enumerate(zip(basic_types, categories, embeddings)):
            bt_str = str(bt).strip()
            cat_str = str(cat).strip()
            
            # Include both BT and CT in ID to ensure uniqueness for each mapping
            bt_id_part = bt_str.lower().replace(' ', '_').replace('/', '_')
            ct_id_part = cat_str.lower().replace(' ', '_').replace('/', '_')
            unique_id = f"bt_{bt_id_part}__{ct_id_part}"
            
            vectors.append({
                "id": unique_id,
                "values": emb,
                "metadata": {
                    "basic_type": bt_str,
                    "category": cat_str,
                    "type": "bt_ct_mapping"
                }
            })
        
        # Upsert batch
        mappings_index.upsert(vectors=vectors)
        total_uploaded += len(vectors)
        print(f"  Uploaded {batch_start + 1}-{batch_end} ({total_uploaded}/{len(bt_ct_df)})")
        
    except Exception as e:
        errors.append(f"Batch {batch_start}: {e}")
        print(f"  ❌ Error in batch {batch_start}: {e}")
    
    # Small delay to avoid rate limits
    time.sleep(0.5)

print(f"\n✅ Uploaded {total_uploaded} BT-CT mappings to {MAPPINGS_INDEX}")
if errors:
    print(f"⚠️ {len(errors)} errors occurred")

Uploading 1605 BT-CT mappings to sku-tagging-mappings...

  ✅ Index already exists: sku-tagging-mappings (dim=3072)
  Uploaded 1-50 (1/1605)
  Uploaded 51-100 (2/1605)
  Uploaded 101-150 (3/1605)
  Uploaded 151-200 (4/1605)
  Uploaded 201-250 (5/1605)
  Uploaded 251-300 (6/1605)
  Uploaded 301-350 (7/1605)
  Uploaded 351-400 (8/1605)
  Uploaded 401-450 (9/1605)
  Uploaded 451-500 (10/1605)
  Uploaded 501-550 (11/1605)
  Uploaded 551-600 (12/1605)
  Uploaded 601-650 (13/1605)
  Uploaded 651-700 (14/1605)
  Uploaded 701-750 (15/1605)
  Uploaded 751-800 (16/1605)
  Uploaded 801-850 (17/1605)
  Uploaded 851-900 (18/1605)
  Uploaded 901-950 (19/1605)
  Uploaded 951-1000 (20/1605)
  Uploaded 1001-1050 (21/1605)
  Uploaded 1051-1100 (22/1605)
  Uploaded 1101-1150 (23/1605)
  Uploaded 1151-1200 (24/1605)
  Uploaded 1201-1250 (25/1605)
  Uploaded 1251-1300 (26/1605)
  Uploaded 1301-1350 (27/1605)
  Uploaded 1351-1400 (28/1605)
  Uploaded 1401-1450 (29/1605)
  Uploaded 1451-1500 (30/1605)
  Uplo

In [ ]:
# =============================================================================
# CELL 7: Load Catalog SKUs from CSV
# =============================================================================

print(f"Loading Catalog SKUs from {CATALOG_CSV_PATH}...")

catalog_df = pd.read_csv(CATALOG_CSV_PATH)
catalog_df.columns = [c.strip() for c in catalog_df.columns]

print(f"  ✅ Loaded {len(catalog_df)} SKUs")
print(f"  Columns: {list(catalog_df.columns)}")
display(catalog_df.head())

In [ ]:
# =============================================================================
# CELL 8: Upload All Catalog SKUs to Pinecone
# =============================================================================
import time
import ast

print(f"Uploading {len(catalog_df)} catalog SKUs to {CATALOG_INDEX}...\n")

catalog_index = ensure_index_exists(CATALOG_INDEX)

# Identify columns
name_col = 'Name' if 'Name' in catalog_df.columns else catalog_df.columns[0]
cat_col = 'Categories' if 'Categories' in catalog_df.columns else None
bt_col = 'Basic Type' if 'Basic Type' in catalog_df.columns else None
gk_col = 'Generic keywords' if 'Generic keywords' in catalog_df.columns else None

print(f"  Using columns: Name={name_col}, Cat={cat_col}, BT={bt_col}, GK={gk_col}")

BATCH_SIZE = 50
total_uploaded = 0
errors = []

for batch_start in range(0, len(catalog_df), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(catalog_df))
    batch_df = catalog_df.iloc[batch_start:batch_end]
    
    # Prepare texts for embedding (SKU names)
    sku_names = batch_df[name_col].astype(str).tolist()
    
    try:
        # Generate embeddings for all SKU names in batch
        embeddings = get_embeddings_batch(sku_names)
        
        # Create vector records
        vectors = []
        for idx, (_, row) in enumerate(batch_df.iterrows()):
            sku_name = str(row[name_col]).strip()
            
            # Parse metadata
            category = str(row[cat_col]).strip() if cat_col and pd.notna(row.get(cat_col)) else ""
            basic_type = str(row[bt_col]).strip() if bt_col and pd.notna(row.get(bt_col)) else ""
            
            # Parse generic keywords (may be a string representation of a list)
            gks = []
            if gk_col and pd.notna(row.get(gk_col)):
                gk_raw = row[gk_col]
                if isinstance(gk_raw, list):
                    gks = gk_raw
                elif isinstance(gk_raw, str) and gk_raw.startswith('['):
                    try:
                        gks = ast.literal_eval(gk_raw)
                    except:
                        gks = [gk_raw]
                else:
                    gks = [str(gk_raw)]
            
            vectors.append({
                "id": f"sku_{hash(sku_name) % 10**10}",
                "values": embeddings[idx],
                "metadata": {
                    "sku_name": sku_name,
                    "category": category,
                    "basic_type": basic_type,
                    "generic_keywords": gks[:20] if gks else [],  # Limit GKs for metadata size
                    "type": "catalog_sku"
                }
            })
        
        # Upsert batch
        catalog_index.upsert(vectors=vectors)
        total_uploaded += len(vectors)
        print(f"  Uploaded {batch_start + 1}-{batch_end} ({total_uploaded}/{len(catalog_df)})")
        
    except Exception as e:
        errors.append(f"Batch {batch_start}: {e}")
        print(f"  ❌ Error in batch {batch_start}: {e}")
    
    # Small delay to avoid rate limits
    time.sleep(0.5)

print(f"\n✅ Uploaded {total_uploaded} catalog SKUs to {CATALOG_INDEX}")
if errors:
    print(f"⚠️ {len(errors)} errors occurred")

In [ ]:
# =============================================================================
# CELL 9: Verify Index Stats
# =============================================================================

print("Index Statistics:\n")

# Mappings index
mappings_index = pc.Index(MAPPINGS_INDEX)
mappings_stats = mappings_index.describe_index_stats()
print(f"📊 {MAPPINGS_INDEX}:")
print(f"   Total vectors: {mappings_stats.total_vector_count}")
print(f"   Dimension: {mappings_stats.dimension}")

# Catalog index
catalog_index = pc.Index(CATALOG_INDEX)
catalog_stats = catalog_index.describe_index_stats()
print(f"\n📊 {CATALOG_INDEX}:")
print(f"   Total vectors: {catalog_stats.total_vector_count}")
print(f"   Dimension: {catalog_stats.dimension}")

print("\n✅ Done!")